# Trend Raster Hotspot Analysis Workflow

- This notebook is meant to be reproducible from within any Pro project on any computer. However, it will not function outside of ArcGIS Pro.
- It is for batch processing multiple rasters with a variety of aggregation and hotspot parameters. 
- It assumes that this notebook has been placed in the project folder, next to the .aprx file.
- Users must specify some parameters in the User Configuration section of the notebook. It should be possible to run the entire workflow without changing any other cells. 

#### Version 1.2

- New version assumes all classification has been done in advance.
- It accepts a list of up to 10 input rasters so you can analyze multiple datasets with the same parameters at the same time.
- The input subfolders have been removed; now all input files go directly inside "inputs".
- The CRS of all input data is automatically checked and reprojected if needed.

## User Instructions

**SCROLL DOWN AND MODIFY THE "USER CONFIGURATION" CELL BEFORE RUNNING**

#### Inputs:
- 1 area of interest as a shapefile, located in the "inputs" folder
- A list of filenames of up to 10 input rasters, located in the "inputs" folder
- A list of ratios of pixels:hexagons you want to try
- A list of numbers of nearest neighbors you want to try using for hotspot analysis
- A coordinate reference system

#### Outputs:
- Output files will use the input filename + suffixes
- Final outputs (aggregated hexagons and hotspots) will be saved to the "outputs" folders
- Intermediate outputs will be saved to the GDB

#### Default folder structure

This tool expects the "inputs" folder to already exist within your project folder. The "outputs" folders will be created if missing. 

In [4]:
# Project/
    # inputs/
    # outputs/
        # aggregation/
        # hotspots/
    # TrendRasterHotspots.ipynb

## Setup

### Import modules

In [5]:
import arcpy
import os
arcpy.CheckOutExtension("Spatial")
import math

### Project context & environment

In [6]:
# Get the Pro project we're currently using
aprx = arcpy.mp.ArcGISProject("CURRENT")

# Setting the project directory for use in creating relative paths later
# Important: this assumes that this notebook is in the project directory!
# change to project directory and then save cwd to variable "project_dir"
os.chdir(aprx.homeFolder)
project_dir = os.getcwd()

# Get the project's default geodatabase, where we'll be saving some output
gdb_path = aprx.defaultGeodatabase

# Report 
print("Project file:", aprx.filePath)
print("Project directory:", project_dir)
print("Workspace:", gdb_path)

Project file: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.aprx
Project directory: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis
Workspace: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.gdb


#### Emergency file path hardcoding

This notebook is meant to automatically detect your project file and build the folder structure around that. However, you can hardcode file paths here if you really need to. 

In [7]:
# your project's absolute path
# aprx = "C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\TrendRasterHotspotAnalysis.aprx"

# Important: this assumes that this notebook is in the project directory!
# os.chdir(aprx.homeFolder)
# project_dir = os.getcwd()
#gdb_path = aprx.defaultGeodatabase
# Report 
#print("Project file:", aprx.filePath)
#print("Project directory:", project_dir)
#print("Workspace:", gdb_path)

#### File path builder helper function

In [8]:
def build_path(base, rel_path):
    return os.path.abspath(os.path.join(base, rel_path))

# demo
example_rel_path = "example_file.tif"
example_abs_path = build_path(project_dir, example_rel_path)
print(example_abs_path)

C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\example_file.tif


## **User Configuration**
- Place AOI shapefile and input rasters in the /YourProjectFolder/inputs/ folder
- Specify filenames below (not full paths)
- Final outputs will be written to /YourProjectFolder/outputs/
- Intermediate outputs will be written to the default GDB

### **EDIT ONLY THE FOLLOWING CELL**

In [9]:
### ====== INPUT FILES ====== ###

# specify area of interest shapefile name, located in /YourProjectFolder/Inputs/
aoi_filename = "MetroArea.shp"

# specify file names of input rasters, located in /YourProjectFolder/Inputs/
# to use only 1 raster, make a 1-item list. 
# Guardrail: limited to <=10 rasters at a time.
# use quotes, separate with commas
input_rasters = ["growMag.tif", "declineMag.tif"]

### ====== ANALYSIS PARAMETERS ====== ###

# coordinate reference system for analysis (rasters will be reprojected if needed)
# Default is UTM Zone 15
target_crs = arcpy.SpatialReference(26915)

# list: ratios of hexagon width to pixel width for aggregation
hex_ratios = [6, 10, 15]

# list: number of nearest neighbors for hotspot analysis
# multiples of 6 are best
neighbors = [3, 6, 12]

### ====== OUTPUT SETTINGS ====== ###

# set whether outputs can overwrite existing data
# RECOMMEND LEAVING "True"
arcpy.env.overwriteOutput = True

### ====== PROJECT FOLDER STRUCTURE ====== ###
# These follow the expected structure. Do not edit unless needed.

inputs_rel_path = "inputs" # input folder name
outputs_rel_path = "outputs" # output folder name
aggregation_rel_path = "aggregation" # aggregation output subfolder name
hotspots_rel_path = "hotspots" # hotspots output subfolder name

## Definitions

In [10]:
# another little helper function for visualization

def add_to_map(layer_path, layer_name=None):
    """
    Explicitly adds a dataset to the active ArcGIS Pro map.
    Only use for outputs you actually want visible.
    """

    aprx = arcpy.mp.ArcGISProject("CURRENT")
    m = aprx.activeMap

    if layer_name is None:
        layer_name = os.path.basename(layer_path)

    try:
        m.addDataFromPath(layer_path)
        print(f"Added to map: {layer_name}")
    except Exception as e:
        print(f"Could not add layer {layer_name}: {e}")

### Step 1: Resolve & Validate

In [11]:
# -----------------------------------------------------------
# Resolve project folder paths
def resolve_paths(project_dir):
    inputs_dir = build_path(project_dir, inputs_rel_path)
    outputs_dir = build_path(project_dir, outputs_rel_path)

    aggregation_dir = build_path(outputs_dir, aggregation_rel_path)
    hotspots_dir = build_path(outputs_dir, hotspots_rel_path)

    return {
        "inputs_dir": inputs_dir,
        "outputs_dir": outputs_dir,
        "aggregation_dir": aggregation_dir,
        "hotspots_dir": hotspots_dir
    }

In [12]:
# -----------------------------------------------------------
# Folder validation and creation
def validate_or_create_folder(folder_path, description, create=True):
    if os.path.isdir(folder_path):
        print(f"{description} folder exists: {folder_path}")
        return

    if create:
        os.makedirs(folder_path, exist_ok=True)
        print(f"[INFO] Created {description} folder: {folder_path}")
    else:
        raise FileNotFoundError(
            f"{description} folder does not exist: {folder_path}"
        )

In [13]:
# -----------------------------------------------------------
# Validate file existence
def validate_file(path, description):
    if not arcpy.Exists(path) and not os.path.isfile(path):
        raise FileNotFoundError(f"{description} file not found: {path}")
    print(f"{description} file found: {path}")


# -----------------------------------------------------------
# Validate AOI is a shapefile
def validate_aoi(aoi_path):
    validate_file(aoi_path, "AOI")

    if not aoi_path.lower().endswith(".shp"):
        raise ValueError(f"AOI must be a shapefile (.shp): {aoi_path}")


# -----------------------------------------------------------
# Validate raster inputs
def validate_rasters(raster_paths):
    if len(raster_paths) > 10:
        raise ValueError("Maximum of 10 rasters allowed.")

    for path in raster_paths:
        validate_file(path, "Raster")

        desc = arcpy.Describe(path)
        if desc.dataType not in ["RasterDataset", "RasterLayer"]:
            raise ValueError(f"Invalid raster dataset: {path}")

In [14]:
# -----------------------------------------------------------
# Validate CRS exists (not Unknown)
def validate_has_crs(dataset, description="Dataset"):
    desc = arcpy.Describe(dataset)
    sr = desc.spatialReference

    if sr is None or sr.name == "Unknown":
        raise ValueError(f"{description} has undefined coordinate system: {dataset}")

    print(f"{description} CRS exists: {sr.name}")

#### Resolve & Validate Wrapper Function

In [15]:
# -----------------------------------------------------------
# MASTER: Validation + Resolution Wrapper
def validate_and_resolve(project_dir):
    
    # --- Resolve folders ---
    paths = resolve_paths(project_dir)

    inputs_dir = paths["inputs_dir"]
    outputs_dir = paths["outputs_dir"]
    aggregation_dir = paths["aggregation_dir"]
    hotspots_dir = paths["hotspots_dir"]

    # --- Validate folders ---
    validate_or_create_folder(inputs_dir, "Inputs", create=False)
    validate_or_create_folder(outputs_dir, "Outputs", create=True)
    validate_or_create_folder(aggregation_dir, "Aggregation", create=True)
    validate_or_create_folder(hotspots_dir, "Hotspots", create=True)

    # --- Resolve file paths ---
    aoi_path = build_path(inputs_dir, aoi_filename)

    raster_paths = [
        build_path(inputs_dir, fname)
        for fname in input_rasters
    ]

    # --- Validate files ---
    validate_aoi(aoi_path)
    validate_rasters(raster_paths)

    # --- Validate CRS exists (but DO NOT enforce match) ---
    validate_has_crs(aoi_path, "AOI")

    for rp in raster_paths:
        validate_has_crs(rp, "Raster")

    # --- Return everything cleanly ---
    return {
        **paths,
        "aoi_path": aoi_path,
        "raster_paths": raster_paths
    }

### Step 2: Prepare Data

In [16]:
# -----------------------------------------------------------
def ensure_aoi_crs(aoi_path, target_crs, gdb_path):
    desc = arcpy.Describe(aoi_path)
    sr = desc.spatialReference

    if sr.factoryCode == target_crs.factoryCode:
        print("AOI already in target CRS.")
        return aoi_path

    print("Reprojecting AOI to target CRS...")
    aoi_projected = build_path(gdb_path, "aoi_projected")

    arcpy.management.Project(
        aoi_path,
        aoi_projected,
        target_crs
    )

    print("AOI reprojected.")
    return aoi_projected

In [17]:
# -----------------------------------------------------------
def ensure_raster_crs(raster_path, target_crs, gdb_path, prefix):
    desc = arcpy.Describe(raster_path)
    sr = desc.spatialReference

    if sr.factoryCode == target_crs.factoryCode:
        print(f"{prefix}: Raster already in target CRS.")
        return raster_path

    print(f"{prefix}: Reprojecting raster...")
    projected_raster = build_path(gdb_path, f"{prefix}_projected")

    arcpy.management.ProjectRaster(
        raster_path,
        projected_raster,
        out_coor_system=target_crs,
        resampling_type="NEAREST"
    )

    print(f"{prefix}: Raster reprojected.")
    return projected_raster

In [18]:
# -----------------------------------------------------------
def clip_raster_to_aoi(raster_path, aoi_path, gdb_path, prefix):
    print(f"{prefix}: Clipping raster to AOI...")

    clipped_raster = build_path(gdb_path, f"{prefix}_clipped")

    arcpy.management.Clip(
        raster_path,
        "#",
        clipped_raster,
        aoi_path,
        "#",
        "ClippingGeometry",
        "MAINTAIN_EXTENT"
    )

    print(f"{prefix}: Clipped raster ready.")
    return clipped_raster

In [19]:
# -----------------------------------------------------------
def prep_single_raster(raster_path, aoi_path, gdb_path, target_crs):
    
    # derive prefix from filename (THIS is your new pattern)
    prefix = os.path.splitext(os.path.basename(raster_path))[0]

    print(f"\n--- Processing: {prefix} ---")

    # Step 1: Ensure raster CRS
    raster_prepped = ensure_raster_crs(
        raster_path, target_crs, gdb_path, prefix
    )

    # Step 2: Clip
    clipped_raster = clip_raster_to_aoi(
        raster_prepped, aoi_path, gdb_path, prefix
    )

    return clipped_raster

#### Wrapper function: projects if needed & clips the input rasters

In [20]:
def prep_data(raster_paths, aoi_path, gdb_path, target_crs):
    
    print("Preparing AOI...")
    aoi_prepped = ensure_aoi_crs(aoi_path, target_crs, gdb_path)

    prepped_rasters = {}

    for rp in raster_paths:

        prefix = os.path.splitext(os.path.basename(rp))[0]

        prepared = prep_single_raster(
            rp,
            aoi_prepped,
            gdb_path,
            target_crs
        )

        prepped_rasters[prefix] = prepared

    return aoi_prepped, prepped_rasters

In [21]:
# Test call

#print("Running test: prepare_trend_raster")

#binary_raster = prepare_trend_raster(raster_paths=raster_paths,aoi_path=aoi_path, gdb_path=gdb_path,name_prefix=name_prefix,out_crs=target_crs)

### Step 3: Aggregate the binary trend data

#### Calculate hexagon areas for each pixel:hexagon width ratio

In [22]:
def hex_area_from_ratio(binary_raster, ratio, prefix=None):
    r_desc = arcpy.Describe(binary_raster)
    pixel_width = round(r_desc.meanCellWidth, 2)

    hex_width = pixel_width * ratio
    hex_side_length = hex_width / 2
    hex_area = (3 * math.sqrt(3) / 2) * (hex_side_length ** 2)

    label = f"[{prefix}] " if prefix else ""

    info = {
        "ratio": ratio,
        "hex_width": hex_width,
        "hex_side_length": hex_side_length,
        "hex_area": hex_area
    }

    print(f"{label}For hexagons {ratio}x the pixel width ({pixel_width}m):")
    print(f"  Hex width (vertex-to-vertex): {hex_width:.2f} m")
    print(f"  Hex area:                     {hex_area:.2f} sq m")
    print("-" * 60)

    return info

In [23]:
# test call

#for ratio in hex_ratios:
#    hex_area_from_ratio(binary_raster, ratio)

#### Create & clip tessellated hexagons for each ratio

In [24]:
def create_and_clip_hexagons(binary_raster, aoi_path, hex_info_list, gdb_path, target_crs, name_prefix):
    """
    Generate hexagon tessellations for each hex info dictionary, clipped to AOI.
    """

    hex_outputs = {}

    for info in hex_info_list:
        ratio = info["ratio"]
        hex_area_val = info["hex_area"]

        print(f"Generating hexagons for ratio {ratio}x pixel width...")

        # output feature class paths
        fc_unclipped = build_path(gdb_path, f"{name_prefix}_hex{ratio}_unclipped")
        fc_clipped = build_path(gdb_path, f"{name_prefix}_hex{ratio}_clipped")

        # generate tessellation
        arcpy.management.GenerateTessellation(
            Output_Feature_Class=fc_unclipped,
            Extent=aoi_path,
            Shape_Type="HEXAGON",
            Size=hex_area_val,
            Spatial_Reference=target_crs
        )

        # clip to AOI
        arcpy.analysis.Clip(
            in_features=fc_unclipped,
            clip_features=aoi_path,
            out_feature_class=fc_clipped
        )

        print(f"Clipped {ratio}x hexagons saved.")

        hex_outputs[ratio] = fc_clipped

    return hex_outputs

In [25]:
# test call

#hex_info_list = [hex_area_from_ratio(binary_raster, r) for r in hex_ratios]

#hex_meshes = create_and_clip_hexagons(binary_raster, aoi_path, hex_info_list, gdb_path, name_prefix)


#### Sum binary raster values and join to hexagon 

In [26]:
def sum_raster_to_hexagons(hex_dict, binary_raster, aggregation_dir, prefix):
    """
    Summarize binary raster values within each hexagon and join to hex polygons.
    """

    aggregated_hexes = {}

    for ratio, hex_fc in hex_dict.items():

        print(f"Summing raster values within {ratio}x hexagons...")

        # output table path
        zonal_table = build_path(
            aggregation_dir,
            f"{prefix}_hex{ratio}_zonal_table.dbf"
        )

        # output aggregated feature class
        aggregated_fc = build_path(
            aggregation_dir,
            f"{prefix}_hex{ratio}_sum"
        )

        # copy features (preserve original hexes)
        arcpy.management.CopyFeatures(hex_fc, aggregated_fc)

        hex_zone_field = arcpy.Describe(aggregated_fc).OIDFieldName

        # zonal statistics
        arcpy.sa.ZonalStatisticsAsTable(
            in_zone_data=aggregated_fc,
            zone_field=hex_zone_field,
            in_value_raster=binary_raster,
            out_table=zonal_table,
            statistics_type="SUM",
            ignore_nodata="DATA"
        )

        # detect join field
        possible_join_fields = [
            f.name for f in arcpy.ListFields(zonal_table)
            if f.name.startswith(hex_zone_field)
        ]

        if not possible_join_fields:
            raise ValueError(
                f"No matching OID field found in zonal table for {hex_zone_field}"
            )

        table_zone_field = possible_join_fields[0]

        # join back to hexagons
        print("Joining sums to hexagons...")

        arcpy.management.JoinField(
            in_data=aggregated_fc,
            in_field=hex_zone_field,
            join_table=zonal_table,
            join_field=table_zone_field,
            fields=["SUM"]
        )

        print(f"Aggregated hexagons saved.")

        aggregated_hexes[ratio] = aggregated_fc

    print("Aggregation complete.")
    return aggregated_hexes

In [27]:
#test call
#aggregated_hexes = sum_raster_to_hexagons(hex_meshes, binary_raster, aggregation_dir, name_prefix)

#### Wrapper function: Hexagon Aggregation

In [28]:
def hexagon_aggregation(
    binary_raster,
    aoi_path,
    hex_ratios,
    gdb_path,
    aggregation_dir,
    target_crs,
    prefix
):
    """
    Full hexagon aggregation pipeline:
    - compute hex sizes
    - generate hex grids
    - aggregate raster values into hexes
    """

    # Step 1: calculate hex geometry
    hex_info_list = [
        hex_area_from_ratio(binary_raster, r)
        for r in hex_ratios
    ]

    # Step 2: create hex meshes
    hex_meshes = create_and_clip_hexagons(
        binary_raster,
        aoi_path,
        hex_info_list,
        gdb_path,
        target_crs,
        prefix
    )

    # Step 3: aggregate raster values
    aggregated_hexes = sum_raster_to_hexagons(
        hex_meshes,
        binary_raster,
        aggregation_dir,
        prefix
    )

    return aggregated_hexes

In [29]:
#test call
# aggregated_hexes = hexagon_aggregation(binary_raster, aoi_path, hex_ratios, gdb_path, name_prefix)

### Step 4: Hotspot analysis

#### Hotspots for a single aggregation

In [30]:
def hotspot_single_analysis(
    in_fc,
    out_fc,
    number_of_neighbors,
    value_field="SUM"
):
    arcpy.stats.HotSpots(
        Input_Feature_Class=in_fc,
        Input_Field=value_field,
        Output_Feature_Class=out_fc,
        Conceptualization_of_Spatial_Relationships="K_NEAREST_NEIGHBORS",
        Distance_Method="EUCLIDEAN_DISTANCE",
        Standardization="ROW",
        Apply_False_Discovery_Rate__FDR__Correction="APPLY_FDR",
        number_of_neighbors=number_of_neighbors
    )

    return out_fc

#### Wrapper function: hotspots for all aggregations. 

In [31]:
def hotspot_analysis(
    aggregated_hexes,
    neighbors,
    hotspots_dir,
    prefix,
    value_field="SUM"
):
    """
    Run hotspot analysis on aggregated hexagon layers.
    """

    hotspot_outputs = {}

    for ratio, aggregated_fc in aggregated_hexes.items():
        for k in neighbors:

            print(f"Running hotspot analysis: {ratio}x hexagons, {k} neighbors...")

            out_fc = build_path(
                hotspots_dir,
                f"{prefix}_hex{ratio}_hotspots_{k}n"
            )

            hotspot_single_analysis(
                in_fc=aggregated_fc,
                out_fc=out_fc,
                number_of_neighbors=k,
                value_field=value_field
            )

            hotspot_outputs[(ratio, k)] = out_fc

            print(f"Hotspot analysis complete for {ratio}x, {k}n.")

    return hotspot_outputs

## Execution

In [32]:
def main():

    print("\n==============================")
    print("STARTING WORKFLOW EXECUTION")
    print("==============================\n")

    # -------------------------------------------------------
    # Step 1: Resolve + validate project
    # -------------------------------------------------------
    resolved = validate_and_resolve(project_dir)

    aoi_path = resolved["aoi_path"]
    raster_paths = resolved["raster_paths"]
    aggregation_dir = resolved["aggregation_dir"]
    hotspots_dir = resolved["hotspots_dir"]

    print("\nProject validated and paths resolved.\n")

    # -------------------------------------------------------
    # Step 2: BATCH PREP (runs once per dataset)
    # -------------------------------------------------------
    aoi_prepped, prepped_rasters = prep_data(
        raster_paths,
        aoi_path,
        gdb_path,
        target_crs
    )

    print("\nBatch preprocessing complete.\n")

    # -------------------------------------------------------
    # Output storage
    # -------------------------------------------------------
    all_outputs = {}

    # -------------------------------------------------------
    # Step 3: PER RASTER LOOP (analysis only)
    # -------------------------------------------------------
    for prefix, prepared_raster in prepped_rasters.items():

        print("\n======================================")
        print(f"PROCESSING: {prefix}")
        print("======================================\n")

        # -----------------------------
        # Hex aggregation
        # -----------------------------
        aggregated_hexes = hexagon_aggregation(
            prepared_raster,
            aoi_prepped,
            hex_ratios,
            gdb_path,
            aggregation_dir,
            target_crs,
            prefix
        )

        # -----------------------------
        # Hotspots
        # -----------------------------
        hotspot_results = hotspot_analysis(
            aggregated_hexes,
            neighbors,
            hotspots_dir,
            prefix,
            value_field="SUM"
        )

        # -----------------------------
        # Store outputs
        # -----------------------------
        all_outputs[prefix] = {
            "prepared_raster": prepared_raster,
            "aggregated_hexes": aggregated_hexes,
            "hotspots": hotspot_results
        }

        print(f"\nCompleted raster: {prefix}\n")

    # -------------------------------------------------------
    # Final summary
    # -------------------------------------------------------
    print("\n==============================")
    print("WORKFLOW COMPLETE")
    print("==============================\n")

    return all_outputs

In [33]:
results = main()


STARTING WORKFLOW EXECUTION

Inputs folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\inputs
Outputs folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\outputs
Aggregation folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\outputs\aggregation
Hotspots folder exists: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\outputs\hotspots
AOI file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\inputs\MetroArea.shp
Raster file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\inputs\growMag.tif
Raster file found: C:\Users\cleor\Documents\ArcGIS\Projects\TrendRasterHotspotAnalysis\inputs\declineMag.tif
AOI CRS exists: NAD_1983_UTM_Zone_15N
Raster CRS exists: NAD_1983_UTM_Zone_15N
Raster CRS exists: NAD_1983_UTM_Zone_15N

Project validated and paths resolved.

Preparing AOI...
AOI already in target CRS.

--- Processing: growMag -